# Documentation de la classe `Location`

La classe `Location` (`kadi.weather.location.Location`) représente une position géographique (latitude, longitude) au sein de la zone d'étude du Bénin. 
Elle sert de composant central pour déduire automatiquement la zone agro-climatique et le régime des pluies.

## 1. Initialisation et Validation GPS
L'initialisation requiert une latitude et une longitude en degrés décimaux. Le système vérifie immédiatement si les coordonnées appartiennent à la `Bounding Box` du Bénin, définie dans la configuration globale de KadiPy. Si ce n'est pas le cas, une exception `LocationNotFound` est levée.

In [1]:
# Changer de répertoire
import os
from pathlib import Path
print(Path.cwd())
new_dir = Path("~/Bureau/kadipy/").expanduser().resolve()
os.chdir(new_dir)
print(Path.cwd())

from kadi.weather.location import Location
from kadi.exceptions import LocationNotFound

# Localisation valide : Cotonou (Sud Bénin)
try:
    loc = Location(latitude=6.3667, longitude=2.4333, name='Cotonou')
    print(f"Initialisation réussie pour : {loc.name}")
except LocationNotFound as e:
    print(e)

# Localisation invalide : Paris
try:
    paris = Location(latitude=48.8566, longitude=2.3522)
except LocationNotFound as e:
    print(f"\nErreur interceptée avec succès : {e}")

/home/dels/Bureau/kadipy/docs/weather
/home/dels/Bureau/kadipy
Initialisation réussie pour : Cotonou

Erreur interceptée avec succès : Les coordonnées GPS (48.8566, 2.3522) sont en dehors de la zone d'étude.


## 2. Détection Automatique de la Zone et du Régime
Lors de la création de l'objet, la méthode `detect_zone()` classe la localisation en trois zones :
- **Sud** : Latitude < 7.5°N
- **Centre** : Latitude entre 7.5°N et 9.0°N
- **Nord** : Latitude > 9.0°N

En parallèle, la méthode interne `_detect_regime()` associe le régime des pluies :
- **Bimodal** pour le Sud (deux saisons des pluies).
- **Unimodal** pour le Centre et le Nord (une seule grande saison).

In [3]:
# Créons 3 localisations pour observer les détections
sud = Location(latitude=6.5, longitude=2.0, name='Ouidah')
centre = Location(latitude=8.5, longitude=2.0, name='Savè')
nord = Location(latitude=10.5, longitude=2.0, name='Natitingou')

for l in [sud, centre, nord]:
    print(f"{l.name:<12} -> Zone: {l.zone:<7} | Régime: {l.climate_regime}")

Ouidah       -> Zone: Sud     | Régime: bimodal
Savè         -> Zone: Centre  | Régime: unimodal
Natitingou   -> Zone: Nord    | Régime: unimodal


## 3. Paramètres Climatiques Locaux
**Méthode :** `get_climate_params() -> dict`

Cette méthode fournit les paramètres algorithmiques adaptés à la région. C'est particulièrement utile pour la classe `Phenology` qui utilise une méthode de calcul d'Onset différente (ex: Sivakumar au Nord, Walter-Anyadike au Sud).

In [5]:
print("Paramètres pour le Nord :", nord.get_climate_params())
print("Paramètres pour le Sud :", sud.get_climate_params())
print("Paramètres pour le centre :", centre.get_climate_params())

Paramètres pour le Nord : {'Tbase': 10, 'onset_method': 'sivakumar'}
Paramètres pour le Sud : {'Tbase': 10, 'onset_method': 'walter_anyadike'}
Paramètres pour le centre : {'Tbase': 10, 'onset_method': 'hybrid'}


## 4. Sérialisation (Export Dictionnaire)
**Méthode :** `to_dict() -> dict`

Afin d'insérer les résultats dans une API JSON ou dans le système de log, `to_dict()` génère un dictionnaire standard des propriétés de la position.

In [8]:
import json

print(json.dumps(centre.to_dict(), indent=2, ensure_ascii=False))

{
  "name": "Savè",
  "lat": 8.5,
  "lon": 2.0,
  "zone": "Centre",
  "regime": "unimodal"
}
